# Manipulation Check

_______________________________________________________________________________________________________________________________

### Imports

In [1]:
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import wilcoxon

### Drop duplicates and sort sessions

In [2]:
messages = pd.read_csv("messages_real8.csv")
sessions = pd.read_csv("sessions_real8.csv")

In [3]:
sessions = sessions.drop_duplicates(subset=["participant_id", "question_index"])
messages = messages.sort_values(["participant_id", "question_index", "id"]).reset_index(drop=True)

## Filter Data

In [4]:
### helper function

In [5]:
def is_complete(pid, sessions_df):
    """Check PID for completeness (completed all debates and ratings)"""
    participant_sessions = sessions_df[sessions_df["participant_id"] == pid]
    
    # check 6 sessions
    if len(participant_sessions) != 6:
        return False
    
    #check 3 aligned and 3 misaligned
    condition_counts = participant_sessions["condition"].value_counts()
    if condition_counts.get("Aligned", 0) != 3:
        return False
    if condition_counts.get("Misaligned", 0) != 3:
        return False
    
    #check missing final ratings
    if participant_sessions["final_rating"].isna().any():
        missing = participant_sessions["final_rating"].isna().sum()
        return False
    
    return True

In [6]:
### Check for participants

In [7]:
all_pids = sessions["participant_id"].unique()

#manual exclude due to technical difficulties
manual_exclude = ["P013"]
all_pids = [pid for pid in all_pids if pid not in manual_exclude]

complete_pids = []
excluded_pids = []

print("Completeness check")
for pid in sorted(all_pids):
    complete = is_complete(pid, sessions)
    status = "Included" if complete else "Excluded"
    if complete:
        complete_pids.append(pid)
    else:
        excluded_pids.append(pid)

print(f"\nIncluded: {len(complete_pids)} participants: {complete_pids}")
print(f"Excluded: {len(excluded_pids)} participants: {excluded_pids}")

Completeness check

Included: 20 participants: ['P002', 'P008', 'P011', 'P012', 'P014', 'P032', 'P036', 'P040', 'P045', 'P047', 'P053', 'P057', 'P061', 'P066', 'P067', 'P068', 'P069', 'P072', 'P075', 'P082']
Excluded: 15 participants: ['P001', 'P015', 'P020', 'P022', 'P027', 'P029', 'P039', 'P042', 'P052', 'P054', 'P058', 'P059', 'P065', 'P076', 'P081']


In [8]:
### Set of complete participates

In [9]:
sessions_clean = sessions[sessions["participant_id"].isin(complete_pids)].copy()
messages_clean = messages[messages["participant_id"].isin(complete_pids)].copy()

## Metrics

In [10]:
### Helpers ###

First quantifyers, personal pronouns and indefinite pronouns were defined because they do not have an explicit tag in spacy.
They are gathered from various sources online, but may not be an exhaustive list

In [11]:
quantifiers = {
    "many", "few", "some", "all", "most", "much",
    "several", "enough", "each", "every", "both",
    "either", "neither", "less", "fewer", "more"
}

personal_pronouns = {
    "i", "me", "my", "mine", "myself",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself",
    "she", "her", "hers", "herself",
    "we", "us", "our", "ours", "ourselves",
    "they", "them", "their", "theirs", "themselves"
}

indefinite_pronouns = {
    "it", "this", "that", "these", "those",
    "someone", "somebody", "something",
    "anyone", "anybody", "anything",
    "everyone", "everybody", "everything",
    "nobody", "nothing",
    "its", "itself", "one", "oneself"
}


def tokenize(text):
    """Tokenize text, removing punct and whitespace."""
    doc = nlp(str(text).lower())
    return [t.text for t in doc if not t.is_punct and not t.is_space]

def lemmatize(text):
    """Lwmatize text, removing puct and whitespace"""
    doc = nlp(str(text).lower())
    return [t.lemma_ for t in doc if not t.is_punct and not t.is_space]

### Linguistic Style Matching (LSM)

For the LSM measure, proportions of linguistic categories were calculated.
Full calculation follows method by Gonzales et al. 2010 [link](https://doi.org/10.1177/0093650209351468)

In [12]:
### Get word category proportions ###

In [13]:
def get_lsm_categories(text):
    """
    Calculate the proportion of each function word category in a text.
    Returns a dict of 9 category proportions.
    """
    doc = nlp(str(text).lower())
    tokens = [t for t in doc if not t.is_punct and not t.is_space]

    categories = {
        "auxiliary_verbs": 0,
        "articles": 0,
        "adverbs": 0,
        "personal_pronouns": 0,
        "indefinite_pronouns": 0,
        "prepositions": 0,
        "conjunctions": 0,
        "quantifiers": 0,
        "negations": 0
    }

    total = len(tokens)
    if total == 0:
        return categories

    for token in tokens:
        word = token.text.lower()

        if token.pos_ == "AUX":
            categories["auxiliary_verbs"] += 1
        if token.pos_ == "DET" and word in {"a", "an", "the"}:
            categories["articles"] += 1
        if token.pos_ == "ADV":
            categories["adverbs"] += 1
        if word in personal_pronouns:
            categories["personal_pronouns"] += 1
        if word in indefinite_pronouns:
            categories["indefinite_pronouns"] += 1
        if token.pos_ == "ADP":
            categories["prepositions"] += 1
        if token.pos_ in {"CCONJ", "SCONJ"}:
            categories["conjunctions"] += 1
        if token.dep_ == "neg" or word in {"not", "never", "no"}:
            categories["negations"] += 1
        if word in quantifiers:
            categories["quantifiers"] += 1

    #counts to proportions
    for key in categories:
        categories[key] = categories[key] / total

    return categories

In [14]:
### calculate scores ###

In [15]:
def calculate_lsm(user_text, ai_text):
    """
    Calculate LSM (Language Style Matching) between user and AI text.
    LSM = average of per-category similarity scores.
    Per-category score = 1 - |a - b| / (a + b), or 1 if both are 0.
    Returns overall LSM score and per-category breakdown.
    """
    user_scores = get_lsm_categories(user_text)
    ai_scores = get_lsm_categories(ai_text)

    category_scores = {}
    for category in user_scores:
        u = user_scores[category]
        a = ai_scores[category]
        if u == 0 and a == 0:
            category_scores[category] = 1.0
        else:
            category_scores[category] = 1 - abs(u - a) / (u + a)

    overall_lsm = sum(category_scores.values()) / len(category_scores)
    return overall_lsm, category_scores

### Local Level Alignment (LLA)

LLA was calculated as word overlap between user text and AI text.
Multiple versions were checked:
- User normalized (divided by length of user text)
- AI normalized (divided by length of AI text)
- Using word
- Using word lemmas

In [16]:
### Calculate word overlap ###

In [17]:
def calculate_lla(user_text, ai_text):
    """
    Calculate LLA (Lexical Level Alignment) in both directions.
    lla_ai: proportion of AI words that also appear in the user's text. 
    lla_user: proportion of unique user words that also appear in the AI's text.
    """
    user_words = tokenize(user_text)
    ai_words = tokenize(ai_text)

    user_set = set(user_words)
    ai_set = set(ai_words)

    lla_ai = len([w for w in ai_words if w in user_set]) / len(ai_words) if ai_words else 0
    lla_user = len(user_set & ai_set) / len(user_set) if user_set else 0

    return lla_ai, lla_user


In [18]:
def calculate_lla_lemmas(user_text, ai_text):
    """
    Calculate LLA in both directions, using lemmas instead of exact words
    lla_ai: proportion of AI lemmas that also appear in the user's text.
    lla_user: proportion of unique user lemmas that also appear in the AI's text
    """
    user_lemmas = lemmatize(user_text)
    ai_lemmas = lemmatize(ai_text)

    user_set = set(user_lemmas)
    ai_set = set(ai_lemmas)

    lla_ai = len([l for l in ai_lemmas if l in user_set]) / len(ai_lemmas) if ai_lemmas else 0
    lla_user = len(user_set & ai_set) / len(user_set) if user_set else 0

    return lla_ai, lla_user

### Style Cosine Similarity

Style cosine similarity was calculated using the same categories as LSM, along with sklearns cosine similarity model

In [19]:
### Extract Style Tokens ###

In [20]:
def extract_style_tokens(text):
    """
    Extract only function/style words from text for style embedding comparison.
    Keeps auxiliary verbs, determiners, pronouns, prepositions,
    conjunctions, adverbs, negations, and quantifiers.
    """
    doc = nlp(str(text).lower())
    style_tokens = []

    for token in doc:
        if token.is_punct or token.is_space:
            continue
        word = token.text.lower()
        if token.pos_ in {"AUX", "DET", "PRON", "ADP", "CCONJ", "SCONJ", "ADV"}:
            style_tokens.append(word)
        elif token.dep_ == "neg":
            style_tokens.append(word)
        elif word in quantifiers:
            style_tokens.append(word)

    return " ".join(style_tokens)

In [21]:
def calculate_style_cosine(user_text, ai_text):
    """
    Calculate cosine similarity between style-word embeddings of user and AI text.
    Returns a value between 0 and 1. Returns 0 if either text has no style tokens.
    """
    user_style = extract_style_tokens(user_text)
    ai_style = extract_style_tokens(ai_text)

    if not user_style.strip() or not ai_style.strip():
        return 0.0

    #string to vector representation
    embeddings = embedding_model.encode([user_style, ai_style])
    return float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])

## Calculate Metrics

Metrics were calculated first on a turn level, then averages to a conversation, and finnally the participant level

In [22]:
### Turn Level ###

In [23]:
nlp = spacy.load("en_core_web_sm")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [24]:
turn_rows = []

for (participant_id, question_index), group in messages_clean.groupby(["participant_id", "question_index"]):
    group = group.sort_values("id")

    last_user_text = None
    turn_number = 0

    for _, row in group.iterrows():
        if row["role"] == "user":
            last_user_text = row["content"]

        elif row["role"] == "assistant" and last_user_text is not None:
            turn_number += 1
            ai_text = row["content"]

            #calculate metrics
            lsm, _ = calculate_lsm(last_user_text, ai_text)
            lla_ai, lla_user = calculate_lla(last_user_text, ai_text)
            lla_ai_lemma, lla_user_lemma = calculate_lla_lemmas(last_user_text, ai_text)
            style_cosine = calculate_style_cosine(last_user_text, ai_text)

            #word counts
            user_word_count = len(tokenize(last_user_text))
            ai_word_count = len(tokenize(ai_text))

            turn_row = {
                "participant_id": participant_id,
                "question_index": question_index,
                "turn_number": turn_number,
                "user_text": last_user_text,
                "ai_text": ai_text,
                "user_style_tokens": extract_style_tokens(last_user_text),
                "ai_style_tokens": extract_style_tokens(ai_text),
                "lsm": lsm,
                "lla_ai": lla_ai,
                "lla_user": lla_user,
                "lla_ai_lemma": lla_ai_lemma,
                "lla_user_lemma": lla_user_lemma,
                "style_cosine": style_cosine,
                "user_word_count": user_word_count,
                "ai_word_count": ai_word_count,
                "length_ratio": ai_word_count / user_word_count if user_word_count > 0 else None,
            }

            turn_rows.append(turn_row)

turn_df = pd.DataFrame(turn_rows)

#combine condition and dilemma info from sessions
turn_df = turn_df.merge(
    sessions_clean[["participant_id", "question_index", "condition", "dilemma_id"]],
    on=["participant_id", "question_index"],
    how="left"
)

In [25]:
### Conversation Level ###

In [26]:
conversation_df = (
    turn_df
    .groupby(["participant_id", "question_index", "condition", "dilemma_id"])
    .agg(
        mean_lsm=("lsm", "mean"),
        mean_lla_ai=("lla_ai", "mean"),
        mean_lla_user=("lla_user", "mean"),
        mean_lla_ai_lemma=("lla_ai_lemma", "mean"),
        mean_lla_user_lemma=("lla_user_lemma", "mean"),
        mean_style_cosine=("style_cosine", "mean"),
        mean_length_ratio=("length_ratio", "mean"),
        turns=("turn_number", "count")
    )
    .reset_index()
)

In [27]:
### Participant Level

In [28]:
participant_df = (
    conversation_df
    .groupby(["participant_id", "condition"])
    .agg(
        mean_lsm=("mean_lsm", "mean"),
        mean_lla_ai=("mean_lla_ai", "mean"),
        mean_lla_user=("mean_lla_user", "mean"),
        mean_lla_ai_lemma=("mean_lla_ai_lemma", "mean"),
        mean_lla_user_lemma=("mean_lla_user_lemma", "mean"),
        mean_style_cosine=("mean_style_cosine", "mean"),
        mean_length_ratio=("mean_length_ratio", "mean"),
        total_turns=("turns", "sum")
    )
    .reset_index()
)

In [29]:
print("\n Participant Level Metrics:")
print(participant_df.head())


 Participant Level Metrics:
  participant_id   condition  mean_lsm  mean_lla_ai  mean_lla_user  \
0           P002     Aligned  0.569690     0.288234       0.666441   
1           P002  Misaligned  0.617536     0.212681       0.269104   
2           P008     Aligned  0.541021     0.303318       0.654769   
3           P008  Misaligned  0.492327     0.188062       0.330551   
4           P011     Aligned  0.573709     0.282770       0.720680   

   mean_lla_ai_lemma  mean_lla_user_lemma  mean_style_cosine  \
0           0.321691             0.713857           0.438546   
1           0.228404             0.306682           0.360459   
2           0.339201             0.705203           0.374080   
3           0.209151             0.369006           0.357446   
4           0.311313             0.745938           0.536104   

   mean_length_ratio  total_turns  
0           9.456014            9  
1           2.325762           11  
2           7.419107            8  
3           6.323954 

In [30]:
### Mean ###

In [31]:
for metric in ["mean_lsm", "mean_lla_ai", "mean_lla_user", "mean_lla_ai_lemma", "mean_lla_user_lemma", "mean_style_cosine"]:
    aligned = participant_df[participant_df["condition"] == "Aligned"][metric].mean()
    misaligned = participant_df[participant_df["condition"] == "Misaligned"][metric].mean()
    print(f"  {metric}: Aligned={aligned:.3f}, Misaligned={misaligned:.3f}, Diff={aligned - misaligned:.3f}")

  mean_lsm: Aligned=0.579, Misaligned=0.523, Diff=0.056
  mean_lla_ai: Aligned=0.305, Misaligned=0.173, Diff=0.132
  mean_lla_user: Aligned=0.622, Misaligned=0.315, Diff=0.307
  mean_lla_ai_lemma: Aligned=0.337, Misaligned=0.193, Diff=0.145
  mean_lla_user_lemma: Aligned=0.665, Misaligned=0.358, Diff=0.307
  mean_style_cosine: Aligned=0.480, Misaligned=0.354, Diff=0.126


## Manipulation Check

In [32]:
metrics = {
    "mean_lsm": "LSM",
    "mean_lla_ai": "LLA (AI-normalized)",
    "mean_lla_user": "LLA (User-normalized)",
    "mean_lla_ai_lemma": "LLA AI normalized, using lemmas",
    "mean_lla_user_lemma": "LLA user-normalized using lemmas",
    "mean_style_cosine": "Style Cosine Similarity"
}


aligned = (
    participant_df[participant_df["condition"] == "Aligned"]
    .set_index("participant_id")
    .add_suffix("_aligned")
)

misaligned = (
    participant_df[participant_df["condition"] == "Misaligned"]
    .set_index("participant_id")
    .add_suffix("_misaligned")
)

wide_df = aligned.join(misaligned, how="inner").reset_index()

print(f"N = {len(wide_df)} participants")
print(f"Participants: {wide_df['participant_id'].tolist()}")

# Wilcoxon Tests

print("Wilcoxon Signed-Rank Test:")
print(f"{'Metric':<30} {'Aligned':>10} {'Misaligned':>12} {'Diff':>8} {'W':>8} {'p-value':>10} {'Sig':>5}")
print("_" * 90)

for col, label in metrics.items():
    aligned_vals = wide_df[f"{col}_aligned"]
    misaligned_vals = wide_df[f"{col}_misaligned"]
    diff = aligned_vals - misaligned_vals

    aligned_mean = aligned_vals.mean()
    misaligned_mean = misaligned_vals.mean()
    diff_mean = diff.mean()

    #stats
    stat, p = wilcoxon(aligned_vals, misaligned_vals)
    sig = "yes" if p < 0.05 else "no"
    p_str = f"{p:.2e}"
    stat_str = f"{stat:.1f}"
    
    print(f"{label:<30} {aligned_mean:>10.3f} {misaligned_mean:>12.3f} {diff_mean:>8.3f} {stat_str:>8} {p_str:>10} {sig:>5}")

print("\nSignificance: p <.05")


N = 20 participants
Participants: ['P002', 'P008', 'P011', 'P012', 'P014', 'P032', 'P036', 'P040', 'P045', 'P047', 'P053', 'P057', 'P061', 'P066', 'P067', 'P068', 'P069', 'P072', 'P075', 'P082']
Wilcoxon Signed-Rank Test:
Metric                            Aligned   Misaligned     Diff        W    p-value   Sig
__________________________________________________________________________________________
LSM                                 0.579        0.523    0.056     31.0   4.22e-03   yes
LLA (AI-normalized)                 0.305        0.173    0.132      0.0   1.91e-06   yes
LLA (User-normalized)               0.622        0.315    0.307      0.0   1.91e-06   yes
LLA AI normalized, using lemmas      0.337        0.193    0.145      0.0   1.91e-06   yes
LLA user-normalized using lemmas      0.665        0.358    0.307      0.0   1.91e-06   yes
Style Cosine Similarity             0.480        0.354    0.126      0.0   1.91e-06   yes

Significance: p <.05
